# 01 - Baselines: the bar every model must clear

**Goal:** define the prediction problem precisely, then score two dumb predictors honestly.
If a model cannot beat these, it has learned nothing.

**The problem (fixed by `SPEC.md`):** after today's data is in, predict **tomorrow night's `sleep_hours`**
(horizon = 1 day). Secondary targets later: `hrv`, `resting_hr`.

**Concepts**
- *Target alignment / leakage*: the label for row `t` is the value at `t+1`. Shift the wrong way and you are
  "predicting" the past with the future. This is the most common beginner bug.
- *Walk-forward evaluation*: at each date `t`, pretend you only know data up to `t`. No random splits, ever.
- *MAE*: average absolute miss, in hours. Easy to feel. `evaluate.mae` skips missing pairs and tells you how many it used.
- *Skill*: `1 - MAE_model / MAE_naive`. Positive = beats "tomorrow = today".

**Baselines**
1. `naive`: tomorrow = today.
2. `mean7`: tomorrow = mean of the last 7 available nights (decide: what if only 3 of 7 exist?).

In [ ]:
# --- Setup (provided, working). Do not edit; fix things in api/app/ml/features.py instead. ---
import pathlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "api").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "api"))

from app.ml import features, evaluate          # the SAME modules the nightly job imports

raw = features.frame_from_csv(ROOT / "ml/data/daily_metrics.csv")   # one row per calendar day
feat = features.build(raw)                                          # baselines, z-scores, missing flags
print(f"{len(raw)} days, {raw.index.min().date()} -> {raw.index.max().date()}, "
      f"{int(raw['row_present'].sum())} with a row, {int(raw['sleep_hours'].isna().sum())} with no sleep value")
raw.tail()

## 1. Build the target
Write `make_target(df, horizon)` so that row `t` holds the value that happens `horizon` days later. Check by eye on 5 rows.

In [ ]:
def make_target(df, horizon=1):
    """TODO: return a Series aligned to df.index where value at t is df['sleep_hours'] at t+horizon."""
    pass

# y = make_target(raw)
# pd.concat([raw["sleep_hours"], y], axis=1).head(8)    # row t target must equal row t+1 value

## 2. Naive and 7-day-mean predictions
Both must use only information available at day `t`. Decide and document how you treat missing days in `mean7`.

In [ ]:
# TODO: pred_naive = ...
# TODO: pred_mean7 = ...   (pandas rolling with min_periods; your choice, write down why)

## 3. Score on the last 30 days
This is the exact window the promotion gate uses. Report MAE, number of scored days, and skill of `mean7` vs `naive`.

In [ ]:
# TODO: pick the last 30 dates that have a real target; use evaluate.mae(y_true, y_pred) and evaluate.skill(...)
# Print a small table: model | MAE (h) | n days | skill vs naive

## 4. Look at the errors, not only the average
Plot the errors over time. Where are the biggest misses? (A 1.6 h night produces a huge naive error the next day: is that a model problem?)

In [ ]:
# TODO: error plot + the 5 worst days.

## Results (fill in)
| Model | MAE (h) | n | Skill vs naive |
|---|---|---|---|
| naive | | | 0 |
| mean7 | | | |

**Reflection:** what MAE would be *useful* to you in practice? Is 0.6 h good enough to act on?